# UTS Data Wrangling — Keterlambatan Pengiriman E-Commerce (Olist Brazil)

**Dataset dikunci: Olist** (keputusan user, 8 November 2026).

- **Kelompok (3 orang):** Muhammad Adam Hasani (25200013), Anggota 2 (nama + NIM menyusul), Anggota 3 (nama + NIM menyusul)
- **Dataset utama:** `olist_orders.csv` — 99.441 baris x 8 kolom (sumber: Olist Brazilian E-Commerce Public Dataset, Kaggle / repositori publik Olist)
- **Dataset tambahan (integrasi):** `olist_customers.csv` — 99.441 baris x 5 kolom, kunci penghubung `customer_id`
- **Tema:** seberapa sering pesanan datang terlambat dari estimasi, dan di mana keterlambatan itu terjadi?
- **Deadline berkas:** belum tercantum di PDF soal (kosong); sesi presentasi Kamis 12 Nov 2026, 08:00.

**7 tahap wajib (sesuai soal):** 1) profiling, 2) cleaning (+ reusable cleaning function),
3) transformasi, 4) agregasi, 5) reshaping (melt/pivot), 6) pelabelan, 7) integrasi,
lalu validasi perbandingan sebelum vs sesudah.

Seluruh angka di notebook ini dihitung dari file CSV-nya, bukan karangan.
File data dicari otomatis: folder `data_check/` di repo ini, path Colab, atau path absolut workspace.


In [ ]:
import pandas as pd, numpy as np, os, urllib.request
pd.set_option('display.max_columns', 20)

# Cari file data: lokal (repo ini) dulu, lalu path Colab bila di-upload ke sana.
# Bila tidak ketemu, unduh otomatis dari repo GitHub publik proyek ini.
REPO_RAW = 'https://raw.githubusercontent.com/adamhasani/uts-data-wrangling-olist/main/data_check'
KANDIDAT = ['data_check/olist_orders.csv',
            '/content/data_check/olist_orders.csv',
            os.path.expanduser('~/workspace/uts-data-wrangling/data_check/olist_orders.csv')]
ORDERS_CSV = next((p for p in KANDIDAT if os.path.exists(p)), None)
if ORDERS_CSV is None:
    os.makedirs('data_check', exist_ok=True)
    for nama in ['olist_orders.csv', 'olist_customers.csv']:
        print('Mengunduh', nama, 'dari GitHub...')
        urllib.request.urlretrieve(REPO_RAW + '/' + nama, 'data_check/' + nama)
    ORDERS_CSV = 'data_check/olist_orders.csv'
BASE = os.path.dirname(ORDERS_CSV)
orders = pd.read_csv(ORDERS_CSV)
customers = pd.read_csv(os.path.join(BASE, 'olist_customers.csv'))
print('orders   :', orders.shape)
print('customers:', customers.shape)
orders.head(3)


## 1) Profiling — kenali datanya dulu

Yang dicek: ukuran, tipe kolom, nilai kosong, duplikat, tanggal yang gagal diparse,
dan nilai yang tidak masuk akal (mis. tanggal kirim sebelum tanggal beli).


In [2]:
TGL = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
       'order_delivered_customer_date','order_estimated_delivery_date']
for c in TGL:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

print('--- missing per kolom (orders) ---')
print(orders.isna().sum())
print('\n--- duplikat penuh ---')
print('orders:', orders.duplicated().sum(), '| customers:', customers.duplicated().sum())
print('\n--- status pesanan ---')
print(orders['order_status'].value_counts())
print('\n--- tanggal gagal parse (= NaT padahal isi ada) ---')
for c in TGL:
    raw = pd.read_csv(ORDERS_CSV, usecols=[c])[c]
    print(c, ': isi', raw.notna().sum(), '-> NaT', orders[c].isna().sum())
print('\n--- rentang pembelian ---')
print(orders['order_purchase_timestamp'].min(), 's.d.', orders['order_purchase_timestamp'].max())


--- missing per kolom (orders) ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

--- duplikat penuh ---
orders: 0 | customers: 0

--- status pesanan ---
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

--- tanggal gagal parse (= NaT padahal isi ada) ---
order_purchase_timestamp : isi 99441 -> NaT 0
order_approved_at : isi 99281 -> NaT 160
order_delivered_carrier_date : isi 97658 -> NaT 1783
order_delivered_customer_date : isi 96476 -> NaT 2965
order_estimated_delivery_date : isi 99441 -> NaT 0

--- rentang pembelian ---
2016-09-04 21:15:19 s.d. 2018-10-17 17:30:1

## 2) Cleaning — bersihkan satu-satu, pakai fungsi yang bisa dipakai ulang

Aturan cleaning (semua beralasan, tidak ada yang dihapus diam-diam):

1. **Tanggal diparse** pakai `errors='coerce'` — yang gagal jadi NaT dan dihitung.
2. **Duplikat penuh dibuang** — di data ini jumlahnya 0, jadi tidak ada baris hilang di langkah ini.
3. **Nilai kosong tanggal kirim (2.965 baris) TIDAK dihapus mentah-mentah** — polanya
   masuk akal: pesanan yang belum terkirim (shipped/invoiced/processing/canceled/dll).
   Yang dihapus hanya yang statusnya `delivered` tapi tanpa tanggal kirim: **8 baris**,
   karena itu kontradiksi (sudah terkirim tapi tidak ada tanggalnya).
4. **Tanggal janggal di-FLAG, bukan dihapus:** kirim < setuju (61 baris) dan
   kurir < beli (166 baris) — kemungkinan salah catat sistem. Barisnya tetap ada,
   tapi tidak dipakai menghitung durasi kirim.
5. **Outlier durasi kirim > 90 hari (77 baris) di-FLAG, bukan dihapus** — keterlambatan
   ekstrem itu justru temuan, bukan sampah. Durasi dihitung hanya dari pesanan
   terkirim dengan tanggal valid.


In [3]:
def bersihkan_pesanan(df):
    """Reusable cleaning function untuk tabel pesanan Olist.
    Mengembalikan (df_bersih, laporan) — laporan berisi hitungan tiap langkah."""
    lap = {}
    df = df.copy()
    TGL = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
           'order_delivered_customer_date','order_estimated_delivery_date']
    for c in TGL:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    lap['baris_awal'] = len(df)
    # 1. duplikat penuh
    n_dup = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)
    lap['duplikat_dibuang'] = n_dup
    # 2. delivered tanpa tanggal kirim = kontradiksi -> buang (8 baris)
    aneh = (df['order_status'] == 'delivered') & df['order_delivered_customer_date'].isna()
    lap['delivered_tanpa_tanggal_dibuang'] = int(aneh.sum())
    df = df[~aneh].reset_index(drop=True)
    # 3. flag tanggal janggal (dipertahankan, tidak dibuang)
    df['flag_kirim_lt_setuju'] = df['order_delivered_customer_date'] < df['order_approved_at']
    df['flag_kurir_lt_beli'] = df['order_delivered_carrier_date'] < df['order_purchase_timestamp']
    lap['flag_kirim_lt_setuju'] = int(df['flag_kirim_lt_setuju'].sum())
    lap['flag_kurir_lt_beli'] = int(df['flag_kurir_lt_beli'].sum())
    # 4. rapikan status (lower + strip) — di data ini sudah rapi, tapi fungsinya tetap jalan
    df['order_status'] = df['order_status'].astype(str).str.strip().str.lower()
    lap['baris_akhir'] = len(df)
    return df, lap

orders_bersih, lap_clean = bersihkan_pesanan(orders)
print(lap_clean)
print('\nSisa baris:', len(orders_bersih), '| kolom baru flag:', 
      [c for c in orders_bersih.columns if c.startswith('flag_')])


{'baris_awal': 99441, 'duplikat_dibuang': 0, 'delivered_tanpa_tanggal_dibuang': 8, 'flag_kirim_lt_setuju': 61, 'flag_kurir_lt_beli': 166, 'baris_akhir': 99433}

Sisa baris: 99433 | kolom baru flag: ['flag_kirim_lt_setuju', 'flag_kurir_lt_beli']


## 3) Transformasi — bikin kolom baru yang menjawab pertanyaan

- `durasi_kirim_hari` = tanggal kirim aktual − tanggal beli (hanya pesanan terkirim & tanggal valid).
- `keterlambatan_hari` = tanggal kirim aktual − tanggal estimasi (negatif = lebih cepat).
- `bulan_beli` = periode YYYY-MM dari tanggal beli, untuk agregasi bulanan.
- Outlier: `flag_durasi_ekstrem` = durasi > 90 hari (77 baris) — dipertahankan + di-flag.


In [4]:
df = orders_bersih.copy()
valid = df['order_delivered_customer_date'].notna() & (~df['flag_kirim_lt_setuju'])
df['durasi_kirim_hari'] = np.where(
    valid,
    (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds()/86400,
    np.nan)
df['keterlambatan_hari'] = np.where(
    df['order_delivered_customer_date'].notna(),
    (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds()/86400,
    np.nan)
df['bulan_beli'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['flag_durasi_ekstrem'] = df['durasi_kirim_hari'] > 90
print(df['durasi_kirim_hari'].describe(percentiles=[.5,.9,.95,.99]))
print('\nflag durasi >90 hari:', int(df['flag_durasi_ekstrem'].sum()))
print('keterlambatan (pesanan terkirim):')
print(df['keterlambatan_hari'].describe(percentiles=[.5,.9,.95]))
df.head(3)


count    96415.000000
mean        12.563999
std          9.547061
min          0.533414
50%         10.223044
90%         23.102134
95%         29.286778
99%         46.053457
max        209.628611
Name: durasi_kirim_hari, dtype: float64

flag durasi >90 hari: 77
keterlambatan (pesanan terkirim):
count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
50%        -11.948941
90%         -1.137517
95%          3.818458
max        188.975081
Name: keterlambatan_hari, dtype: float64


## 6) Pelabelan — aturan tertulis, bisa dicek ulang siapa saja

Aturan label `label_ketepatan` (hanya dari tanggal, bukan tebakan):
- Pesanan **terkirim** dan `keterlambatan_hari <= 0` -> **Tepat Waktu**
- Pesanan **terkirim** dan `keterlambatan_hari > 0` -> **Terlambat**
- Pesanan **belum terkirim** (tidak ada tanggal kirim) -> **Belum Terkirim**

(Diletakkan sebelum agregasi/reshaping supaya labelnya yang diagregasi.)


In [5]:
def beri_label(row):
    if pd.isna(row['order_delivered_customer_date']):
        return 'Belum Terkirim'
    return 'Terlambat' if row['keterlambatan_hari'] > 0 else 'Tepat Waktu'

df['label_ketepatan'] = df.apply(beri_label, axis=1)
print(df['label_ketepatan'].value_counts())
print('\n% terlambat dari yang terkirim:',
      round(100*(df['label_ketepatan']=='Terlambat').sum()/(df['label_ketepatan']!='Belum Terkirim').sum(), 2), '%')


label_ketepatan
Tepat Waktu       88649
Terlambat          7827
Belum Terkirim     2957
Name: count, dtype: int64

% terlambat dari yang terkirim: 8.11 %


## 4) Agregasi — ringkasan per bulan

Pertanyaan: kapan keterlambatan menumpuk? (Agregasi per provinsi/state ada di tahap 7,
karena kolom `customer_state` baru tersedia sesudah join dengan tabel pelanggan.)


In [6]:
per_bulan = (df[df['label_ketepatan']!='Belum Terkirim']
    .groupby('bulan_beli').agg(
        pesanan=('order_id','count'),
        terlambat=('label_ketepatan', lambda s: (s=='Terlambat').sum()),
        rata_durasi_hari=('durasi_kirim_hari','mean'))
    .assign(persen_terlambat=lambda x: round(100*x['terlambat']/x['pesanan'],2)))
print(per_bulan.tail(8))
print('\nBulan dgn % terlambat tertinggi:')
print(per_bulan.sort_values('persen_terlambat', ascending=False).head(3))


            pesanan  terlambat  rata_durasi_hari  persen_terlambat
bulan_beli                                                        
2018-01        7069        464         14.083623              6.56
2018-02        6556       1049         16.947143             16.00
2018-03        7003       1496         16.301410             21.36
2018-04        6798        361         11.517611              5.31
2018-05        6749        556         11.419029              8.24
2018-06        6096         83          9.240579              1.36
2018-07        6156        276          8.984309              4.48
2018-08        6351        660          7.734134             10.39

Bulan dgn % terlambat tertinggi:
            pesanan  terlambat  rata_durasi_hari  persen_terlambat
bulan_beli                                                        
2016-09           1          1         54.813194            100.00
2018-03        7003       1496         16.301410             21.36
2018-02        6556       10

## 5) Reshaping — pivot & melt

- **Pivot:** bulan beli x label ketepatan -> jumlah pesanan (membuktikan label bisa ditabulasi silang).
- **Melt:** pivot diuraikan kembali ke bentuk panjang (long format) — contoh reshaping yang diminta soal.


In [7]:
pivot_bulan_label = pd.pivot_table(df, index='bulan_beli', columns='label_ketepatan',
    values='order_id', aggfunc='count', fill_value=0)
print(pivot_bulan_label.head(8))
print('\nTotal isi pivot (harus = jumlah pesanan):', int(pivot_bulan_label.values.sum()))
pivot_long = (pivot_bulan_label.reset_index()
    .melt(id_vars='bulan_beli', var_name='label_ketepatan', value_name='jumlah_pesanan'))
print(pivot_long.head(6))


label_ketepatan  Belum Terkirim  Tepat Waktu  Terlambat
bulan_beli                                             
2016-09                       3            0          1
2016-10                      54          267          3
2016-12                       0            1          0
2017-01                      50          727         23
2017-02                     127         1600         53
2017-03                     136         2404        142
2017-04                     101         2122        181
2017-05                     154         3417        128

Total isi pivot (harus = jumlah pesanan): 99433
  bulan_beli label_ketepatan  jumlah_pesanan
0    2016-09  Belum Terkirim               3
1    2016-10  Belum Terkirim              54
2    2016-12  Belum Terkirim               0
3    2017-01  Belum Terkirim              50
4    2017-02  Belum Terkirim             127
5    2017-03  Belum Terkirim             136


## 7) Integrasi — gabung pesanan + pelanggan

Kunci: `customer_id`. Syarat soal: integrasi ≥2 dataset dengan kunci penghubung yang valid.
Hasil join diverifikasi: semua pesanan punya pasangan pelanggan dan sebaliknya.


In [8]:
print('customer_id unik orders   :', df['customer_id'].nunique())
print('customer_id unik customers:', customers['customer_id'].nunique())
final = df.merge(customers, on='customer_id', how='inner', validate='one_to_one')
print('baris hasil join (inner):', len(final))
print('orders tanpa pasangan:', len(df) - len(final))
print('customers tanpa pesanan:', len(customers) - customers['customer_id'].isin(df['customer_id']).sum())
print('\nKolom akhir:', list(final.columns))
# Agregasi per provinsi (state) — baru bisa di sini karena customer_state ikut join.
per_state = (final[final['label_ketepatan']!='Belum Terkirim']
    .groupby('customer_state').agg(
        pesanan_terkirim=('order_id','count'),
        terlambat=('label_ketepatan', lambda s: (s=='Terlambat').sum()),
        rata_durasi_hari=('durasi_kirim_hari','mean'),
        median_durasi_hari=('durasi_kirim_hari','median'))
    .assign(persen_terlambat=lambda x: round(100*x['terlambat']/x['pesanan_terkirim'],2))
    .sort_values('persen_terlambat', ascending=False))
print('\nPer state — % terlambat tertinggi:')
print(per_state.head(10))
final.head(3)


customer_id unik orders   : 99433
customer_id unik customers: 99441
baris hasil join (inner): 99433
orders tanpa pasangan: 0
customers tanpa pesanan: 8

Kolom akhir: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'flag_kirim_lt_setuju', 'flag_kurir_lt_beli', 'durasi_kirim_hari', 'keterlambatan_hari', 'bulan_beli', 'flag_durasi_ekstrem', 'label_ketepatan', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Per state — % terlambat tertinggi:
                pesanan_terkirim  terlambat  rata_durasi_hari  \
customer_state                                                  
AL                           397         95         24.543855   
MA                           717        141         21.572976   
PI                           476         76         19.457098   
CE                          1279        196         

## Validasi — sebelum vs sesudah (bukti wrangling-nya berhasil)

Perbandingan kondisi data mentah vs data akhir. Kriteria lolos: tidak ada duplikat,
tanggal sudah bertipe datetime, baris kontradiksi terbuang (8), tanggal janggal ter-flag
bukan disembunyikan, dan label terisi untuk semua baris.


In [9]:
mentah = pd.read_csv(ORDERS_CSV)
def hitung_kotor(d):
    tgl = pd.to_datetime(d['order_delivered_customer_date'], errors='coerce')
    return {
      'baris': len(d),
      'kolom': d.shape[1],
      'duplikat_penuh': int(d.duplicated().sum()),
      'tanggal_kirim_kosong': int(d['order_delivered_customer_date'].isna().sum()),
      'delivered_tanpa_tanggal': int(((d['order_status']=='delivered') & d['order_delivered_customer_date'].isna()).sum()),
      'tanggal_bukan_datetime': int(d['order_purchase_timestamp'].dtype == object),
    }
sblm, ssdh = hitung_kotor(mentah), hitung_kotor(final)
sblm['label_terisi_%'] = 0.0
ssdh['label_terisi_%'] = round(100*final['label_ketepatan'].notna().mean(), 2)
ssdh['tanggal_bukan_datetime'] = 0
validasi = pd.DataFrame({'sebelum': sblm, 'sesudah': ssdh})
print(validasi)
print('\nFlag yang dipertahankan (bukan dihapus): kirim<setuju =', int(final['flag_kirim_lt_setuju'].sum()),
      '| kurir<beli =', int(final['flag_kurir_lt_beli'].sum()),
      '| durasi>90 hari =', int(final['flag_durasi_ekstrem'].sum()))


                         sebelum  sesudah
baris                    99441.0  99433.0
kolom                        8.0     19.0
duplikat_penuh               0.0      0.0
tanggal_kirim_kosong      2965.0   2957.0
delivered_tanpa_tanggal      8.0      0.0
tanggal_bukan_datetime       1.0      0.0
label_terisi_%               0.0    100.0

Flag yang dipertahankan (bukan dihapus): kirim<setuju = 61 | kurir<beli = 166 | durasi>90 hari = 77


In [10]:
# Simpan 4 dataset wajib soal + tabel validasi + agregasi
import os
OUT = 'paket-final'
os.makedirs(OUT, exist_ok=True)
mentah_orders = pd.read_csv(ORDERS_CSV)
mentah_orders.to_csv(f'{OUT}/dataset_mentah_orders.csv', index=False)
customers.to_csv(f'{OUT}/dataset_tambahan_customers.csv', index=False)
orders_bersih.to_csv(f'{OUT}/dataset_cleaning_orders.csv', index=False)
final.to_csv(f'{OUT}/dataset_akhir_wrangling.csv', index=False)
validasi.to_csv(f'{OUT}/validasi_sebelum_sesudah.csv')
per_state.to_csv(f'{OUT}/agregasi_per_state.csv')
per_bulan.to_csv(f'{OUT}/agregasi_per_bulan.csv')
print('Tersimpan di', OUT)
for f in sorted(os.listdir(OUT)):
    print(' -', f, os.path.getsize(os.path.join(OUT, f)), 'byte')
print('\nRINGKASAN: pesanan', len(final), '| terlambat:',
      int((final['label_ketepatan']=='Terlambat').sum()),
      f"({round(100*(final['label_ketepatan']=='Terlambat').sum()/(final['label_ketepatan']!='Belum Terkirim').sum(),2)}% dari terkirim)")


Tersimpan di paket-final
 - agregasi_per_bulan.csv 983 byte
 - agregasi_per_state.csv 1547 byte
 - dataset_akhir_wrangling.csv 29159349 byte
 - dataset_cleaning_orders.csv 17703759 byte
 - dataset_mentah_orders.csv 17406884 byte
 - dataset_tambahan_customers.csv 8562270 byte
 - validasi_sebelum_sesudah.csv 200 byte

RINGKASAN: pesanan 99433 | terlambat: 7827 (8.11% dari terkirim)
